In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Cloud AI AlphaGenome Batch Inference Pipeline
<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/alphagenome/cloudai_alphagenome_batch_pipeline.ipynb">
      <img src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fcommunity%2Falphagenome%2Fcloudai_alphagenome_batch_pipeline.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/alphagenome/cloudai_alphagenome_batch_pipeline.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/alphagenome/cloudai_alphagenome_batch_pipeline.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

# Overview

This notebook demonstrates how to run **AlphaGenome** predictions in batch on JSONL input files stored in Google Cloud Storage (GCS) using **Vertex AI Pipelines**.

It uses the pre-compiled pipeline template ([`batch_pipeline/alphagenome_batch_inference_pipeline.yaml`](batch_pipeline/alphagenome_batch_inference_pipeline.yaml)) and sample input manifest ([`batch_pipeline/sample_batch_requests.jsonl`](batch_pipeline/sample_batch_requests.jsonl)), which support three endpoint modes:

* **Existing Shared Endpoint (`existing_endpoint`)**: Connects to your deployed Vertex AI endpoint over gRPC.
* **Existing Dedicated Endpoint (`existing_vertex_ai_url`)**: Connects to your dedicated Vertex AI endpoint URL over HTTPS.
* **Temporary GPU Endpoint (both left blank)**: Deploys a temporary Vertex AI GPU endpoint for the batch run and automatically deletes it when the pipeline completes.

## Install Dependencies

In [ ]:
! pip install --quiet --upgrade google-cloud-aiplatform google-cloud-storage pandas pyarrow anndata numpy

## Authentication

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

# Optional service account email used for GCS access and Vertex AI Pipeline execution.
# Ensure this service account has `roles/aiplatform.user` on the project and
# `roles/storage.objectAdmin` on your GCS bucket.
service_account = ""  # @param {type:"string"}
service_account = service_account.strip() or None

## Configure Project, GCS Bucket, and Endpoint

Provide your Google Cloud project ID, region, and GCS bucket. Then configure **one** of the following endpoint options:

* Set `existing_endpoint` for a **Shared Vertex AI Endpoint** (gRPC), or
* Set `existing_vertex_ai_url` for a **Dedicated Vertex AI Endpoint** (HTTPS), or
* Leave both blank to provision a temporary GPU endpoint for the duration of the run.

In [ ]:
project_id = ""  # @param {type:"string"}
region = "us-west1"  # @param {type:"string"}
gcs_bucket = ""  # @param {type:"string"}

# Option 1: Existing Shared Endpoint resource name (gRPC)
# Format: "projects/<PROJECT_NUMBER>/locations/<REGION>/endpoints/<ENDPOINT_ID>"
existing_endpoint = ""  # @param {type:"string"}

# Option 2: Existing Dedicated Endpoint URL (HTTPS)
# Format: "https://<ENDPOINT_DNS>/v1/projects/<PROJECT_NUMBER>/locations/<REGION>/endpoints/<ENDPOINT_ID>:streamRawPredict"
existing_vertex_ai_url = ""  # @param {type:"string"}

# Model fold selection and concurrency
model_version = "ALL_FOLDS"  # @param ["ALL_FOLDS", "FOLD_0", "FOLD_1", "FOLD_2", "FOLD_3"]
workers = 8  # @param {type:"integer"}

if not project_id or not gcs_bucket:
    raise ValueError("Please set both `project_id` and `gcs_bucket` before running.")

bucket_uri = (
    gcs_bucket.rstrip("/")
    if gcs_bucket.startswith("gs://")
    else f"gs://{gcs_bucket.strip('/')}"
)
bucket_name = bucket_uri[len("gs://") :].split("/", 1)[0]

input_gcs_uri = f"{bucket_uri}/alphagenome_batch/input/sample_batch_requests.jsonl"
output_gcs_uri = f"{bucket_uri}/alphagenome_batch/output/run_01/"
pipeline_root = f"{bucket_uri}/alphagenome_batch/pipeline_root"

print(f"Input GCS URI:  {input_gcs_uri}")
print(f"Output GCS URI: {output_gcs_uri}")
print(f"Pipeline root:  {pipeline_root}")

## Imports

In [ ]:
import io
import json
import tempfile
import urllib.request
from pathlib import Path

import anndata
import google.auth
import numpy as np
import pandas as pd
from google.auth import impersonated_credentials
from google.cloud import aiplatform, storage

## Prepare Pipeline Template and Sample Input File

If you opened this notebook from a local clone of `vertex-ai-samples`, the files in `batch_pipeline/` are used directly. Otherwise, they are downloaded automatically from GitHub.

In [ ]:
GITHUB_BASE_URL = (
    "https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/"
    "main/notebooks/community/alphagenome/batch_pipeline"
)

local_dir = Path("batch_pipeline")
pipeline_yaml_path = local_dir / "alphagenome_batch_inference_pipeline.yaml"
sample_jsonl_path = local_dir / "sample_batch_requests.jsonl"

if not pipeline_yaml_path.exists() or not sample_jsonl_path.exists():
    local_dir.mkdir(parents=True, exist_ok=True)
    if not pipeline_yaml_path.exists():
        urllib.request.urlretrieve(
            f"{GITHUB_BASE_URL}/alphagenome_batch_inference_pipeline.yaml",
            pipeline_yaml_path,
        )
    if not sample_jsonl_path.exists():
        urllib.request.urlretrieve(
            f"{GITHUB_BASE_URL}/sample_batch_requests.jsonl",
            sample_jsonl_path,
        )

sample_rows = [
    json.loads(line)
    for line in sample_jsonl_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"Loaded {len(sample_rows)} sample requests from {sample_jsonl_path}:")
pd.DataFrame(
    [
        {
            "request_id": row.get("request_id"),
            "request_type": row.get("request_type"),
            "organism": row.get("organism", "HOMO_SAPIENS"),
        }
        for row in sample_rows
    ]
)

## Upload Sample Input File to Google Cloud Storage

In [ ]:
scopes = ["https://www.googleapis.com/auth/cloud-platform"]
credentials, _ = google.auth.default(scopes=scopes)
if service_account:
    credentials = impersonated_credentials.Credentials(
        source_credentials=credentials,
        target_principal=service_account,
        target_scopes=scopes,
    )

storage_client = storage.Client(project=project_id, credentials=credentials)
bucket = storage_client.bucket(bucket_name)

input_blob_path = input_gcs_uri[len(f"gs://{bucket_name}/") :]
bucket.blob(input_blob_path).upload_from_filename(
    str(sample_jsonl_path), content_type="application/jsonl"
)
print(f"Uploaded {sample_jsonl_path} -> {input_gcs_uri}")

## Submit the Vertex AI Batch Inference Pipeline

Submit the compiled pipeline YAML to Vertex AI Pipelines and wait for the run to complete. You can also click the Cloud Console link printed below to monitor each step visually.

In [ ]:
aiplatform.init(
    project=project_id,
    location=region,
    staging_bucket=bucket_uri,
)

job = aiplatform.PipelineJob(
    display_name="alphagenome-batch-inference-demo",
    template_path=str(pipeline_yaml_path),
    pipeline_root=pipeline_root,
    enable_caching=False,
    parameter_values={
        "input_gcs_uri": input_gcs_uri,
        "output_gcs_uri": output_gcs_uri,
        "region": region,
        "existing_endpoint": existing_endpoint,
        "existing_vertex_ai_url": existing_vertex_ai_url,
        "model_version": model_version,
        "workers": workers,
    },
)

job.submit(service_account=service_account)
job.wait()

## Inspect Batch Summary and Per-Request Results

In [ ]:
output_prefix = output_gcs_uri[len(f"gs://{bucket_name}/") :]

summary = json.loads(
    bucket.blob(f"{output_prefix}summary.json").download_as_text(encoding="utf-8")
)
print(json.dumps(summary, indent=2))

predictions = [
    json.loads(line)
    for line in bucket.blob(f"{output_prefix}predictions.jsonl")
    .download_as_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]
pd.DataFrame(
    [
        {
            "request_id": rec["request_id"],
            "request_type": rec["request_type"],
            "status": rec["status"],
            "latency_sec": rec.get("latency_sec"),
            "artifact_uris": rec.get("artifact_uris"),
        }
        for rec in predictions
    ]
)

## Load Track Arrays and Variant Scores from GCS

Track predictions are saved as compressed NumPy `.npz` archives (with companion `_metadata.parquet` tables), and variant/interval scores are saved as `.h5ad` `AnnData` files (with flattened `_tidy.parquet` tables when `save_tidy_scores=True`).

In [ ]:
# 1. Load compressed NumPy track predictions (.npz)
npz_bytes = bucket.blob(
    f"{output_prefix}tracks/cyp2b6_predict_interval.npz"
).download_as_bytes()
with np.load(io.BytesIO(npz_bytes), allow_pickle=False) as tracks_npz:
    for key in tracks_npz.files:
        print(f"Track array '{key}': shape={tracks_npz[key].shape}, dtype={tracks_npz[key].dtype}")

# 2. Load AnnData variant score matrix (.h5ad)
with tempfile.TemporaryDirectory() as tmp_dir:
    local_h5ad = Path(tmp_dir) / "apobec3b_score_variant.h5ad"
    bucket.blob(
        f"{output_prefix}scores/apobec3b_score_variant.h5ad"
    ).download_to_filename(str(local_h5ad))
    score_adata = anndata.read_h5ad(local_h5ad)
    print(f"AnnData score matrix shape: {score_adata.shape}")

# 3. Preview flattened tidy variant scores (.parquet)
tidy_bytes = bucket.blob(
    f"{output_prefix}scores/apobec3b_score_variant_tidy.parquet"
).download_as_bytes()
tidy_df = pd.read_parquet(io.BytesIO(tidy_bytes))
tidy_df.head()

# Conclusion

You have run an end-to-end AlphaGenome batch inference pipeline on Vertex AI and loaded the resulting track and variant score files from Google Cloud Storage.

For additional details on pipeline parameters and Google Cloud Console submission, see the [Batch Inference Pipeline guide](batch_pipeline/README.md) and the [interactive quickstart notebook](cloudai_alphagenome_vai_quickstart.ipynb).